# Conversion for Mobile Deployment


- PyTorch → ONNX (in Prototype 10)
- ONNX → TensorFlow → TFLite + INT8 Quantization

References
- https://github.com/DanieliusKr/onnx-example/blob/main/onnx_example.ipynb
- https://www.geeksforgeeks.org/machine-learning/convert-pytorch-model-to-tf-lite-with-onnx-tf/

### Import

In [1]:
# Check Python version
!python --version

Python 3.12.12


In [2]:
try:
    import google.colab
    IN_COLAB = True

    !pip install onnx2tf
    """
    !pip install onnx
    !pip install onnx_tf

    !pip install tensorflow-addons
    !pip install tensorflow-probability
    """
    from google.colab import files
except ImportError:
    IN_COLAB = False

In [3]:
import torch as torch
import torch.nn as nn
import os
import random
import numpy as np
import cv2
import pandas as pd
import scipy.stats as ss

import onnx2tf
import tensorflow as tf
import torch.onnx
import onnx
# from onnx_tf.backend import prepare

## Conversion to TensorFlow and TFLite

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HEIGHT = 128
WIDTH = 128

def data_gen():
    # Initialize dataset folder
    dataset = "Dataset_AIGD"
    if IN_COLAB:
      !unzip -q '/content/Dataset_AIGD.zip' -d '/content/'

      dataset = os.path.join("/content", dataset)
      if os.path.exists(dataset):
        print("Dataset unzipped.")

    return get_input(DEVICE, False, dataset)

def get_input(DEVICE, isRandom, data) -> torch.Tensor:
    """ Generate random input data with the same shape as the model's expected input """
    if isRandom:
        # Generate random video
        # Timesteps, Channels, Height, Width
        video_tensor = torch.rand(20, 3, 1, 1).to(DEVICE)
        # Generate random intent
        intent_direction = random.randint(0, 2)
        # Generate random intent position
        intent_position = random.randint(0, 9)

        # Convert video to frames
        frames = []
        for i in range(video_tensor.shape[0]):
            frame_tensor = video_tensor[i]
            frame_tensor= get_intent_torch(intent_position, i, intent_direction, frame_tensor, 1)
            frames.append(frame_tensor)

        video_tensor = torch.stack(frames, dim=0)
        # Adds batch dimension to the tensor -> Batch, Timesteps, Channels, Height, Width
        video_tensor = video_tensor.unsqueeze(0)

        return video_tensor
    else:
        if IN_COLAB:
          npy_file = "/content/tflite_int8_data.npy"
        else:
          npy_file = "tflite_int8_data.npy"
        representative_data = []

        """
        data_train = os.path.join(data, "train")
        data_train_vid = os.path.join(data_train, "videos")
        data_train_lbl = os.path.join(data_train, "labels")
        labels = [f for f in os.listdir(data_train_lbl) if f.endswith('.csv')]
        """
        data_train_vid = os.path.join(data, "videos")
        data_train_lbl = os.path.join(data, "labels")
        labels = [f for f in os.listdir(data_train_lbl) if f.endswith('.csv')]

        print(f"Number of files in {data_train_vid}: {len(os.listdir(data_train_vid))}")
        for i, video_name in enumerate(os.listdir(data_train_vid)):
        # Representative dataset for Integer Quantization
            if i < 100:
                intent_position = get_intent_position()
                df = pd.read_csv(os.path.join(data_train_lbl, labels[i]))
                intent = get_intent(intent_position, df)

                video_path = os.path.join(data_train_vid, video_name)
                cap = cv2.VideoCapture(video_path)
                frames = []

                for i in range(20):
                    ret, frame = cap.read()
                    if not ret:
                        frame_tensor = torch.zeros((3, HEIGHT, WIDTH))
                    else:
                        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                        """
                        if self.transforms:
                            frame = Image.fromarray(frame)
                            frame_tensor = self.transforms(frame)
                        else:
                        """
                        frame_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0

                    frame_tensor = get_intent_torch(intent_position, i, intent, frame_tensor, 128)
                    frames.append(frame_tensor)

                cap.release()
                video_tensor = torch.stack(frames, dim=0)

                # Global Average Pooling to fit the 1x1 dimensions
                global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
                video_tensor = global_avg_pool(video_tensor)
                representative_data.append(video_tensor)

        print(f"Number of items in the representative_data: {len(representative_data)}")
        # Each item in the list is [20, 6, 1, 1] -> Final is [81, 20, 6, 1, 1]
        tensor_dataset = torch.stack(representative_data, dim=0)
        # Convert to Float32
        float_dataset = tensor_dataset.detach().cpu().numpy().astype(np.float32)
        print(f"Float dataset shape: {float_dataset.shape}")

        m = onnx.load("convlstm.onnx")
        # Print the input shape recorded in the ONNX file
        shape = [d.dim_value for d in m.graph.input[0].type.tensor_type.shape.dim]
        print(f"Onnx input shape: {shape}")

        try:
            # Save the dataset to a numpy file
            np.save(str(npy_file), float_dataset)

        except Exception as e:
            print(f"Error in saving numpy file: {e}")

        if os.path.exists(npy_file):
          print(f"The calibration file is successfully created at {npy_file}. ✅")
        else:
          print(f"The calibration file is not created.")

        np_data = [["video_input", npy_file, [[[[0, 0, 0, 0, 0, 0]]]], [[[[255, 255, 255, 1, 1, 1]]]]]]
        return np_data

def get_intent_position():
    # 50% of the dataset have intent
    if random.random() < 0.5:
        # The time positions of the first 1 second (10 fps) because the intent is placed for a second
        start_frame = 0
        end_frame = 9
        median = (start_frame + end_frame)/2
        range_zero = np.arange(-median, median)

        # Obtain the probability of selecting a timestamp using the adjacent 0.5 areas
        smaller_range = range_zero - 0.5
        higher_range = range_zero + 0.5

        # Probability is the difference of the probability of higher range and lower range
        probability = ss.norm.cdf(higher_range) - ss.norm.cdf(smaller_range)

        # Normalize the probabilities
        # Each probability in probability range is divided by the sum of the probabilities in probability range
        probability /= probability.sum()

        # Select a timestamp based on the probabilities
        range = np.arange(start_frame, end_frame)
        intent_position = np.random.choice(range, p=probability)
    else:
        intent_position = -1

    return intent_position

def get_intent(intent_position, df):
        # Check if the data has no intent
        if intent_position != -1:
            intent = get_label(df)
        else:
            intent = -1
        return intent

def get_label(df):
    """Extract label from CSV using majority vote on last 24 frames."""
    col = 'label_id_corrected' if 'label_id_corrected' in df.columns else df.columns[-1]
    counts = df[col].tail(24).value_counts() # CSV files are in 24 fps

    label_counts = [counts.get(i, 0) for i in range(3)]

    if label_counts[0] == 24:
        return 0  # Front
    elif label_counts[1] > label_counts[2]:
        return 1  # Left
    elif label_counts[1] < label_counts[2]:
        return 2  # Right
    else:
        return int(df[col].tail(12).mode().iloc[0])

def get_intent_torch(intent_position, i, intent, frame_tensor, size):
    # Create a tensor for the intent with the same spatial dimensions as the video frames
    # Used for no intent
    intent_torch = torch.zeros((3, size, size))
    # If intent exists, add intent in its intent position for 1 second (10 frames)
    if intent_position != -1 and intent_position <= i and (intent_position + 10) > i:
        # Convert intent to one-hot vector by filling the specified channel with 1
        intent_torch[intent, :, :] = 1

    intent_torch = intent_torch.to(frame_tensor.device)
    # Append the intent as a channel to the video frame
    frame_tensor = torch.cat((frame_tensor, intent_torch), dim=0)

    return frame_tensor

In [10]:
onnx_path = "convlstm.onnx"
tf_dir = "tf_model"

# Convert from ONNX to TensorFlow and TFLite
onnx2tf.convert(
    input_onnx_file_path=onnx_path,
    output_folder_path=tf_dir,
    copy_onnx_input_output_names_to_tflite=True,
    keep_shape_absolutely_input_names=["video_input"], # Keeps the order of input
    # output_dynamic_range_quantized_tflite=True,
    output_integer_quantized_tflite = True,
    # output_integer_quantized_tflite=self.args.int8,
    custom_input_op_name_np_data_path=data_gen(),
    non_verbose=False,
)
print(f"Success: conversion to Tensorflow. File is saved at: {tf_dir}")

Streaming output truncated to the last 5000 lines.
INFO: onnx_op_type: Conv onnx_op_name: node_conv2d_12
INFO:  input_name.1: cat_12 shape: [1, 70, 1, 1] dtype: float32
INFO:  input_name.2: convlstm.cell_list.0.conv.weight shape: [256, 70, 3, 3] dtype: float32
INFO:  input_name.3: convlstm.cell_list.0.conv.bias shape: [256] dtype: float32
INFO:  output_name.1: conv2d_12 shape: [1, 256, 1, 1] dtype: float32
INFO: tf_op_type: convolution_v2
INFO:  input.1.input: name: tf.concat_58/concat:0 shape: (1, 1, 1, 70) dtype: <dtype: 'float32'> 
INFO:  input.2.weights: shape: (3, 3, 70, 256) dtype: <dtype: 'float32'> 
INFO:  input.3.bias: shape: (256,) dtype: <dtype: 'float32'> 
INFO:  input.4.strides: val: [1, 1] 
INFO:  input.5.dilations: val: [1, 1] 
INFO:  input.6.padding: val: SAME 
INFO:  input.7.group: val: 1 
INFO:  output.1.output: name: tf.math.add_156/Add:0 shape: (1, 1, 1, 256) dtype: <dtype: 'float32'> 

INFO: 180 / 567
INFO: onnx_op_type: Split onnx_op_name: node_Split_210
INFO:  in

In [6]:
"""
# Outdated method for conversion to TensorFlow
onnx_path = "convlstm.onnx"
tf_dir = "tf_model"

onnx_model = onnx.load(onnx_path)
tf_model = prepare(onnx_model)

os.makedirs(tf_dir, exist_ok=True)
tf_model.export_graph(tf_dir)
print(f"Success: conversion to Tensorflow. File is saved at: {tf_dir}")
"""

'\n# Outdated method for conversion to TensorFlow\nonnx_path = "convlstm.onnx"\ntf_dir = "tf_model"\n\nonnx_model = onnx.load(onnx_path)\ntf_model = prepare(onnx_model)\n\nos.makedirs(tf_dir, exist_ok=True)\ntf_model.export_graph(tf_dir)\nprint(f"Success: conversion to Tensorflow. File is saved at: {tf_dir}")\n'

In [7]:
"""
# Convert the TensorFlow model to TFLite model
converter = tf.lite.TFLiteConverter.from_saved_model(tf_dir)
# Dynamic Range Quantization: INT 8 Quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Save the TFLite model
with open ("convlstm.tflite", "wb") as f:
    f.write(tflite_model)

tflite_path = os.path.join(tf_dir, "convlstm.tflite")
print(f"Success: conversion to TFLite. File is saved at: {tflite_path}")
"""

'\n# Convert the TensorFlow model to TFLite model\nconverter = tf.lite.TFLiteConverter.from_saved_model(tf_dir)\n# Dynamic Range Quantization: INT 8 Quantization\nconverter.optimizations = [tf.lite.Optimize.DEFAULT]\ntflite_model = converter.convert()\n\n# Save the TFLite model\nwith open ("convlstm.tflite", "wb") as f:\n    f.write(tflite_model)\n\ntflite_path = os.path.join(tf_dir, "convlstm.tflite")\nprint(f"Success: conversion to TFLite. File is saved at: {tflite_path}")\n'

In [8]:
if IN_COLAB:
    # Automated Download TensorFlow and TFLite models
    if os.path.exists(tf_dir):
        !zip -r 'tf_model.zip' '/content/tf_model'
        files.download("tf_model.zip")
        print("TensorFlow and TFLite models downloaded ✅")
    else:
        print("TensorFlow and TFLite models were not downloaded.")

  adding: content/tf_model/ (stored 0%)
  adding: content/tf_model/convlstm_dynamic_range_quant.tflite (deflated 21%)
  adding: content/tf_model/variables/ (stored 0%)
  adding: content/tf_model/variables/variables.data-00000-of-00001 (deflated 84%)
  adding: content/tf_model/variables/variables.index (deflated 33%)
  adding: content/tf_model/convlstm_float16.tflite (deflated 20%)
  adding: content/tf_model/saved_model.pb (deflated 9%)
  adding: content/tf_model/schema_generated.py (deflated 92%)
  adding: content/tf_model/assets/ (stored 0%)
  adding: content/tf_model/convlstm_float32.tflite (deflated 14%)
  adding: content/tf_model/schema.fbs (deflated 71%)
  adding: content/tf_model/fingerprint.pb (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

TensorFlow and TFLite models downloaded ✅


In [9]:
# Verifying the TFLite model
tflite_path = os.path.join(tf_dir, "convlstm_float16.tflite")
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

output_details = interpreter.get_output_details()[0] # Model has single output.
input_details = interpreter.get_input_details()[0] # Model has single input.
input_shape = input_details['shape']

# Input shape: Batch size, Channels, Height, Width, Timesteps
# Input_data: Batch size, Timesteps, Channels, Height, Width
input_data = get_input(DEVICE, True, None).cpu().numpy().astype(np.float32)
# input_tflite = np.transpose(input_data, (0, 2, 3, 4, 1))
interpreter.set_tensor(input_details['index'], input_data)
interpreter.invoke()

output = interpreter.get_tensor(output_details['index'])
output_shape = output.shape

print("Interpreted TFLite model")
print(f"Input shape: {input_shape}")
print(f"Output shape: {output_shape}")
print(f"Output: {output}")

Interpreted TFLite model
Input shape: [ 1 20  6  1  1]
Output shape: (1, 3)
Output: [[0.07498233 0.01933772 0.06109498]]
